# Fase 2: Tratamento, Normalização e Cálculo do IVS Multidimensional

Este notebook aplica as regras metodológicas para transformar a base bruta num índice analítico de 0 a 1.
**Etapas do Processo:**
1. **Tratamento de Sigilo:** Conversão do marcador "X" do IBGE para `-1`.
2. **Filtro de Elegibilidade (`Dados_sig`):** Classificação dos setores em `OK`, `SIGILOSO`, `COLETIVO` e `ZERADO`.
3. **Cálculo de Proporções:** Divisão dos riscos (falta de água, esgoto, analfabetismo) pelos seus denominadores exatos.
4. **Cálculo da Densidade Habitacional Manual:** `(População de Casas + População de Tendas) / Total de Responsáveis`.
5. **Proxy de Extrema Pobreza:** Composição multidimensional utilizando Renda Média Invertida (40%), Precariedade Sanitária (20%), Moradia Improvisada (20%) e Sobrecarga Demográfica (20%).
6. **Transparência:** Preservação e renomeação dos denominadores para a validação da banca.

In [6]:
import pandas as pd
import numpy as np

print("1. Lendo a Base Bruta Multidimensional...")
caminho_bd = '../../banco_de_dados/'
df = pd.read_csv(caminho_bd + 'Base_Bruta_Multidimensional_Censo2022.csv', sep=';', dtype=str)

print("2. Tratando os marcadores de sigilo do IBGE ('X')...")
# Substitui 'X' por -1 para identificação algorítmica
df = df.replace(['X', 'x'], -1)

# Lista de colunas que devem permanecer como texto
colunas_texto = ['CD_SETOR', 'NM_MUN', 'NM_BAIRRO', 'SITUACAO', 'Moradia_Predominante']

# Converte todas as outras colunas para numérico
colunas_numericas = [col for col in df.columns if col not in colunas_texto]
for col in colunas_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print("Dados convertidos com sucesso!")

1. Lendo a Base Bruta Multidimensional...
2. Tratando os marcadores de sigilo do IBGE ('X')...
Dados convertidos com sucesso!


In [ ]:
print("3. Aplicando as regras de exclusão metodológica (Dados_sig)...")  # Mensagem de status

# Cálculo de Domicílios Coletivos = Responsáveis - Permanentes - Improvisados
df['dom_coletivos_calc'] = df['V01042'] - df['V00001'] - df['V00002']  # Calcula o número de domicílios coletivos
df['dom_coletivos_calc'] = df['dom_coletivos_calc'].clip(lower=0)  # Garante que não haja valores negativos

# Percentual de Domicílios Coletivos no setor
df['perc_coletivos'] = np.where(df['V01042'] > 0, (df['dom_coletivos_calc'] / df['V01042']) * 100, 0)  # Calcula o percentual de coletivos

# Regras estritas da Orientadora para a coluna Dados_sig
condicoes = [
    (df['v0001'] == -1) | (df['V00001'] == -1) | (df['V01042'] == -1),  # Contém Sigilo Base
    (df['perc_coletivos'] >= 100),                                      # 100% Coletivo (Asilos/Presídios)
    (df['v0001'] == 0)                                                  # População Zerada
]
escolhas = ['SIGILOSO', 'COLETIVO', 'ZERADO']  # Define os rótulos para cada condição

df['Dados_sig'] = np.select(condicoes, escolhas, default='OK')  # Aplica as regras e cria a coluna Dados_sig

print("Resumo da Elegibilidade dos Setores no Brasil:")  # Mensagem de status
print(df['Dados_sig'].value_counts())  # Mostra o resumo dos tipos de setores

# Isolamos apenas os setores aprovados para a matemática do IVS
df_ok = df[df['Dados_sig'] == 'OK'].copy()  # Filtra apenas setores OK
# Purificação: transforma o sigilo residual das subcategorias em ZERO 
# para não criar cálculos matemáticos negativos nas frações seguintes.
df_ok = df_ok.replace(-1, 0)  # Substitui -1 por 0 em todo o DataFrame filtrado

3. Aplicando as regras de exclusão metodológica (Dados_sig)...
Resumo da Elegibilidade dos Setores no Brasil:
Dados_sig
OK          450088
ZERADO        9327
SIGILOSO      8684
Name: count, dtype: int64


In [8]:
print("4. Calculando as proporções de risco clássicas (0 a 1)...")

# A. Saneamento e Lixo (Denominador = Total de Responsáveis V01042)
df_ok['ind_agua_inadequada'] = df_ok[['V00112', 'V00113', 'V00114', 'V00115', 'V00116', 'V00117', 'V00118']].sum(axis=1) / df_ok['V01042']
df_ok['ind_esgoto_inadequado'] = df_ok[['V00312', 'V00313', 'V00314', 'V00315', 'V00316']].sum(axis=1) / df_ok['V01042']
df_ok['ind_lixo_inadequado'] = df_ok[['V00398', 'V00399', 'V00400', 'V00401', 'V00402']].sum(axis=1) / df_ok['V01042']

# B. Educação (Denominador = População 15+)
df_ok['ind_analfabetismo'] = np.where(df_ok['V00900'] > 0, df_ok['V00901'] / df_ok['V00900'], 0)

# C. Vulnerabilidade Social / Demografia da Raça (Denominador = População Total v0001)
df_ok['ind_cor_raca'] = df_ok[['V01318', 'V01320', 'V01321']].sum(axis=1) / df_ok['v0001']

# D. Densidade Habitacional Manual (O que a professora pediu)
# (Moradores em Dom. Permanentes + Moradores em Dom. Improvisados) / Total de Responsáveis
df_ok['razao_moradores_domicilio'] = (df_ok['V00005'] + df_ok['V00006']) / df_ok['V01042']

# Normalizando a Densidade Habitacional para escala de 0 a 1
min_dens = df_ok['razao_moradores_domicilio'].min()
max_dens = df_ok['razao_moradores_domicilio'].max()
df_ok['ind_densidade_habitacional'] = (df_ok['razao_moradores_domicilio'] - min_dens) / (max_dens - min_dens)

# Garante limite matemático de 1.0 para os índices base
colunas_clip = ['ind_agua_inadequada', 'ind_esgoto_inadequado', 'ind_lixo_inadequado', 'ind_analfabetismo', 'ind_cor_raca', 'ind_densidade_habitacional']
for col in colunas_clip:
    df_ok[col] = df_ok[col].clip(upper=1.0)

print("Dimensões clássicas processadas!")

4. Calculando as proporções de risco clássicas (0 a 1)...
Dimensões clássicas processadas!


In [9]:
print("5. Construindo o Sub-Índice de Extrema Pobreza Multidimensional...")

# Pilar 1: Renda Normalizada Invertida (40%)
renda_valida = df_ok[df_ok['V06004'] > 0]['V06004']
min_renda = renda_valida.min()
max_renda = renda_valida.max()

df_ok['proxy_renda_invertida'] = np.where(
    df_ok['V06004'] > 0,
    (max_renda - df_ok['V06004']) / (max_renda - min_renda),
    1.0 # Setores sem renda ganham vulnerabilidade máxima
)

# Pilar 2: Precariedade Sanitária / Falta de Banheiro (20%)
df_ok['proxy_falta_banheiro'] = (df_ok['V00236'] + df_ok['V00238']) / df_ok['V01042']

# Pilar 3: Habitação Improvisada / Tendas (20%)
df_ok['proxy_dom_improvisados'] = df_ok['V00002'] / df_ok['V01042']

# Pilar 4: Sobrecarga Demográfica Infantil 0 a 14 anos (20%)
df_ok['proxy_sobrecarga_infantil'] = (df_ok['V01031'] + df_ok['V01032'] + df_ok['V01033']) / df_ok['v0001']

# Limita proxies a 1.0
for col in ['proxy_falta_banheiro', 'proxy_dom_improvisados', 'proxy_sobrecarga_infantil']:
    df_ok[col] = df_ok[col].clip(upper=1.0)

# A FÓRMULA FINAL DO SUB-ÍNDICE ECONÔMICO
df_ok['ind_pobreza_multidimensional'] = (
    (df_ok['proxy_renda_invertida'] * 0.40) +
    (df_ok['proxy_falta_banheiro'] * 0.20) +
    (df_ok['proxy_dom_improvisados'] * 0.20) +
    (df_ok['proxy_sobrecarga_infantil'] * 0.20)
)

print("Sub-Índice calculado e balanceado com sucesso!")

5. Construindo o Sub-Índice de Extrema Pobreza Multidimensional...
Sub-Índice calculado e balanceado com sucesso!


In [ ]:
print("6. Renomeando denominadores para transparência e exportando...")

# Criando as colunas claras para a prova real da banca
df_ok['DENOM_Total_Lares'] = df_ok['V01042']
df_ok['DENOM_Pop_15_Mais'] = df_ok['V00900']
df_ok['DENOM_Pop_Total'] = df_ok['v0001']

# Seleção estruturada das colunas finais
colunas_exportacao = [
    'CD_SETOR', 'NM_MUN', 'NM_BAIRRO', 'SITUACAO', 'Moradia_Predominante', 'Dados_sig',
    'DENOM_Total_Lares', 'DENOM_Pop_Total', 'DENOM_Pop_15_Mais',
    'ind_agua_inadequada', 'ind_esgoto_inadequado', 'ind_lixo_inadequado',
    'ind_analfabetismo', 'ind_cor_raca', 'ind_densidade_habitacional',
    'ind_pobreza_multidimensional'
]

df_final = df_ok[colunas_exportacao].copy()

# Exporta a Base Analítica Limpa e Calculada
caminho_csv_final = caminho_bd + 'Base_Analitica_Multidimensional_Calculada.csv'
df_final.to_csv(caminho_csv_final, index=False, sep=';', encoding='utf-8-sig')

# Exporta também a Base Completa (com os setores ZERADOS e SIGILOSOS) para auditoria se necessário
df.to_csv(caminho_bd + 'Base_Auditoria_Todos_Setores.csv', index=False, sep=';', encoding='utf-8-sig')

print(f"TRABALHO DE EXCELÊNCIA CONCLUÍDO! A base final calculada está em:\n{caminho_csv_final}")

6. Renomeando denominadores para transparência e exportando...
TRABALHO DE EXCELÊNCIA CONCLUÍDO! A base final calculada está em:
../../banco_de_dados/Base_Analitica_Multidimensional_Calculada.csv


: 